# Cat vs Dog CNN — Improved (from-scratch)

**What changed vs the original, at a glance:**

1. **Bug fix:** `self.dropout` is now actually called inside `forward()` — in the original it was defined but never used.
2. **Conv blocks redesigned:** `stride=1, padding=1` so the convolution captures features at full resolution; only `MaxPool` downsamples. Four blocks with channels `32 → 64 → 128 → 256`.
3. **Global Average Pooling** replaces the flatten-then-huge-FC step. This drops millions of redundant parameters and reduces overfitting.
4. **Classifier widened:** the bottleneck went from `10` neurons to `128`.
5. **Input normalization** (ImageNet mean/std) added to every transform pipeline — including the single-image prediction function (otherwise inference inputs look nothing like training inputs).
6. **`.convert('RGB')`** added in the dataset to handle grayscale / RGBA images safely.
7. **Augmentation:** added `ColorJitter` on top of horizontal flip.
8. **LR scheduler:** `ReduceLROnPlateau` halves the learning rate when val loss stops improving.
9. **History tracking + best-model checkpoint:** we save the weights from the epoch with the best validation accuracy and restore them at the end.
10. **Loss/accuracy curves** plotted, **confusion matrix + classification report** on the validation set.
11. **Test loader:** `shuffle=False` (so test order is deterministic).
12. **Minor:** `.item()` to detach losses (no graph leak), `num_workers=2, pin_memory=True` on DataLoaders, `random_state` on the train/val split.


## 1. Setup & imports

In [ ]:
import os
import zipfile
import kagglehub
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
base_path = kagglehub.competition_download('dogs-vs-cats')
print(f"Downloaded zip file at: {base_path}")

train_zip = os.path.join(base_path, 'train.zip')
test_zip  = os.path.join(base_path, 'test1.zip')

print("Unzipping...")
os.makedirs('data', exist_ok=True)
with zipfile.ZipFile(train_zip, 'r') as zip_ref:
    zip_ref.extractall('data')
with zipfile.ZipFile(test_zip, 'r') as zip_ref:
    zip_ref.extractall('data')
print("Done.")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torchvision import transforms
from torch.utils.data import DataLoader, Dataset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, glob, copy, random
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

## 2. Hyperparameters & device

I bumped `epochs` from 10 → 15 and reduced `batch_size` from 100 → 64. These are minor — 64 is just a more conventional power-of-2 batch size, and the extra epochs give the LR scheduler room to step down. Best-model checkpointing means there's no penalty to training longer.

In [ ]:
lr            = 1e-3
batch_size    = 64       # was 100
epochs        = 15       # was 10
dropout_rate  = 0.4      # NEW — passed into the model
num_workers   = 2        # NEW — parallel data loading

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

torch.manual_seed(1234)
if device == 'cuda':
    torch.cuda.manual_seed_all(1234)

## 3. File lists & sample preview

In [ ]:
train_dir = 'data/train'
test_dir  = 'data/test1'

train_list = glob.glob(os.path.join(train_dir, '*.jpg'))
test_list  = glob.glob(os.path.join(test_dir,  '*.jpg'))
print(f"Train images: {len(train_list)}, Test images: {len(test_list)}")

In [ ]:
random_idx = np.random.randint(0, len(train_list), size=10)

fig = plt.figure(figsize=(12, 5))
for i, idx in enumerate(random_idx, start=1):
    ax = fig.add_subplot(2, 5, i)
    img = Image.open(train_list[idx])
    ax.imshow(img)
    ax.axis('off')
plt.show()

## 4. Transforms — **CHANGED**

Two important additions: **normalization** (using ImageNet mean/std), and **ColorJitter** as extra augmentation. We use the *same* `normalize` for train / val / test / single-image inference — this is non-negotiable. If you forget it during inference, the model will see inputs that look nothing like what it trained on and predictions become garbage.

In [ ]:
# ImageNet mean/std — standard even when training from scratch.
# Centering inputs near zero makes optimisation more stable.
normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225],
)

train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),  # less aggressive than default
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),  # NEW
    transforms.ToTensor(),
    normalize,                                                              # NEW
])

# Val/test: NO augmentation, just deterministic resize + normalize.
eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize,
])

## 5. Dataset — **CHANGED**

Two fixes: `.convert('RGB')` so grayscale / RGBA images don't crash the conv layer, and the test-image case (where the filename is just a number like `1.jpg`) now returns an `int` id instead of a string. That makes the downstream sort cleaner.

In [ ]:
class CatDogDataset(Dataset):
    def __init__(self, file_list, transform=None):
        self.file_list = file_list
        self.transform = transform

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_path = self.file_list[idx]
        img = Image.open(img_path).convert('RGB')        # CHANGED: force 3 channels
        if self.transform is not None:
            img = self.transform(img)

        filename = os.path.basename(img_path).split('.')[0]
        if filename == 'dog':
            label = 1
        elif filename == 'cat':
            label = 0
        else:
            # test images are named like '1.jpg' — return the id as label
            label = int(filename)
        return img, label

In [ ]:
train_list, val_list = train_test_split(train_list, test_size=0.2, random_state=1234)  # CHANGED: random_state

train_data = CatDogDataset(train_list, transform=train_transforms)
val_data   = CatDogDataset(val_list,   transform=eval_transforms)
test_data  = CatDogDataset(test_list,  transform=eval_transforms)

In [ ]:
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=True)
val_loader   = DataLoader(val_data,   batch_size=batch_size, shuffle=False,   # CHANGED
                          num_workers=num_workers, pin_memory=True)
test_loader  = DataLoader(test_data,  batch_size=batch_size, shuffle=False,   # CHANGED
                          num_workers=num_workers, pin_memory=True)

print(f"train batches: {len(train_loader)}, val batches: {len(val_loader)}, test batches: {len(test_loader)}")

## 6. The CNN — **major rewrite**

**Old design problem:** every block did `Conv(stride=2)` *and* `MaxPool(2)`, shrinking the spatial size by 4× per block. From a 224×224 input the feature map collapsed to **3×3** after just three blocks. The network had no chance to learn fine textures.

**New design:** `Conv(stride=1, padding=1)` preserves resolution inside the block; the single `MaxPool(2)` is the only downsampler. That gives a gentle 2× per block:

```
input:    224×224×3
block1 →  112×112×32
block2 →   56×56×64
block3 →   28×28×128
block4 →   14×14×256
GAP    →     1×1×256   ← parameter-free spatial collapse
fc1    →   256 → 128
fc2    →   128 → 2
```

**About Global Average Pooling (GAP):** instead of flattening 14×14×256 = 50,176 features into the FC layer (which would create ~6M parameters, dwarfing the convs), we average each channel's 14×14 map to a single number. That gives 256 features. It's parameter-free, acts as strong regularization, and is what every modern CNN since ResNet does.

In [ ]:
class Cnn(nn.Module):
    def __init__(self, dropout=0.4):
        super().__init__()
        self.block1 = self._conv_block(3,   32)
        self.block2 = self._conv_block(32,  64)
        self.block3 = self._conv_block(64,  128)
        self.block4 = self._conv_block(128, 256)

        self.gap = nn.AdaptiveAvgPool2d(1)   # collapses HxW -> 1x1, leaves channels intact

        self.fc1     = nn.Linear(256, 128)
        self.dropout = nn.Dropout(dropout)
        self.fc2     = nn.Linear(128, 2)

    @staticmethod
    def _conv_block(in_ch, out_ch):
        # Conv keeps the spatial size (stride 1, padding 1).
        # MaxPool is the only thing that downsamples.
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.gap(x)             # (B, 256, 1, 1)
        x = x.flatten(1)            # (B, 256)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)         # FIX: actually applied now
        x = self.fc2(x)
        return x

In [ ]:
model     = Cnn(dropout=dropout_rate).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

# NEW: scheduler halves the LR when val loss plateaus for 2 epochs in a row.
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

n_params = sum(p.numel() for p in model.parameters())
print(f"Total trainable parameters: {n_params:,}")
print(model)

## 7. Training loop — **CHANGED**

Three meaningful additions:

1. `history` dict captures train/val loss & accuracy each epoch so we can plot curves.
2. `best_state` snapshots the weights at the epoch with the highest val accuracy. We restore them at the end. This is a poor-man's early stopping — you keep training the full schedule, but only the best version of the model survives.
3. `scheduler.step(val_loss)` after every epoch, and we print the current LR so you can see when it drops.

Also: `.item()` on the loss/accuracy accumulators. Without it, you're keeping a reference to a tensor that's still part of the autograd graph — a slow memory leak on GPU.

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0
best_state   = None

for epoch in range(epochs):
    # -------- Train --------
    model.train()
    epoch_loss, epoch_acc = 0.0, 0.0
    for data, label in train_loader:
        data, label = data.to(device), label.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, label)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() / len(train_loader)                            # .item() detaches
        epoch_acc  += (output.argmax(1) == label).float().mean().item() / len(train_loader)

    # -------- Validate --------
    model.eval()
    val_loss, val_acc = 0.0, 0.0
    with torch.no_grad():
        for data, label in val_loader:
            data, label = data.to(device), label.to(device)
            output = model(data)
            loss = criterion(output, label)
            val_loss += loss.item() / len(val_loader)
            val_acc  += (output.argmax(1) == label).float().mean().item() / len(val_loader)

    # -------- Bookkeeping --------
    history['train_loss'].append(epoch_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(epoch_acc)
    history['val_acc'].append(val_acc)

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())
        marker = "  <-- best"
    else:
        marker = ""

    print(f"Epoch {epoch+1:2d}/{epochs} | "
          f"train loss {epoch_loss:.4f} acc {epoch_acc:.4f} | "
          f"val loss {val_loss:.4f} acc {val_acc:.4f} | "
          f"lr {current_lr:.1e}{marker}")

# Restore the best weights
model.load_state_dict(best_state)
print(f"\nBest val accuracy: {best_val_acc:.4f}")

## 8. Training curves — **NEW**

This is what your lecturer will actually look at. The shape of these curves tells you whether the model was:

- **Underfitting** — both curves flat and high loss → model too small, train longer, or LR too low.
- **Overfitting** — train keeps dropping but val flattens or rises → add regularization, more augmentation, or stop earlier.
- **Just right** — both curves drop together and plateau together.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['val_loss'],   label='val')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss'); axes[0].set_title('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history['train_acc'], label='train')
axes[1].plot(history['val_acc'],   label='val')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('accuracy'); axes[1].set_title('Accuracy')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Confusion matrix on validation set — **NEW**

For a balanced binary task like this, accuracy is fine as a headline number, but the confusion matrix shows *which way* the errors go. If the model systematically calls dogs "cat" but rarely the other way around, you'd see it here.

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for data, label in val_loader:
        data = data.to(device)
        preds = model(data).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(label.numpy())

cm = confusion_matrix(all_labels, all_preds)
print("Confusion matrix:")
print("           pred_cat  pred_dog")
print(f"true_cat   {cm[0,0]:>8}  {cm[0,1]:>8}")
print(f"true_dog   {cm[1,0]:>8}  {cm[1,1]:>8}")
print()
print(classification_report(all_labels, all_preds, target_names=['cat', 'dog']))

## 10. Test predictions & submission

In [ ]:
dog_probs = []
model.eval()
with torch.no_grad():
    for data, fileid in test_loader:
        data = data.to(device)
        preds = model(data)
        preds_list = F.softmax(preds, dim=1)[:, 1].tolist()
        # fileid is an int tensor now (because dataset returns int(filename))
        dog_probs += list(zip(fileid.tolist(), preds_list))

dog_probs.sort(key=lambda x: x[0])   # no string-to-int conversion needed

submission = pd.DataFrame(dog_probs, columns=['id', 'label'])
submission.to_csv('result.csv', index=False)
submission.head()

In [ ]:
# Quick visual sanity check
class_ = {0: 'cat', 1: 'dog'}
fig, axes = plt.subplots(2, 5, figsize=(20, 8))

for ax in axes.ravel():
    i = random.choice(submission['id'].values)
    prob = submission.loc[submission['id'] == i, 'label'].values[0]
    pred = 1 if prob > 0.5 else 0

    img_path = os.path.join(test_dir, f'{i}.jpg')
    img = Image.open(img_path)
    ax.set_title(f"{class_[pred]} ({prob:.2f})")
    ax.imshow(img); ax.axis('off')
plt.show()

## 11. Single-image prediction — **CHANGED**

Critical fix: the inference transform now applies the same `normalize` as training. Without that the model sees pixel values in `[0, 1]` instead of the centered distribution it was trained on, and predictions become unreliable.

In [ ]:
def predict_and_show_image(image_path, model, device):
    model.eval()
    infer_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        normalize,                          # CHANGED: must match training
    ])

    img_original = Image.open(image_path).convert('RGB')
    img_tensor   = infer_transform(img_original).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(img_tensor)
        probs  = F.softmax(output, dim=1)[0]
        cat_prob, dog_prob = probs[0].item() * 100, probs[1].item() * 100

    if dog_prob > cat_prob:
        label, confidence = "DOG", dog_prob
    else:
        label, confidence = "CAT", cat_prob

    plt.figure(figsize=(6, 6))
    plt.imshow(img_original)
    color = 'green' if confidence > 80.0 else 'darkorange'
    plt.title(f"Prediction: {label} ({confidence:.2f}%)",
              fontsize=16, fontweight='bold', color=color, pad=15)
    plt.axis('off'); plt.show()

# Example:
# predict_and_show_image("thong.jpg", model, device)

## 12. Save the trained model

In [ ]:
from google.colab import drive
import json

drive.mount('/content/drive')
save_path = '/content/drive/MyDrive/DogCat_Project'
os.makedirs(save_path, exist_ok=True)

model_file = os.path.join(save_path, 'cnn_model_v2.pth')
torch.save(model.state_dict(), model_file)
print(f"Saved model weights to: {model_file}")

# Save config alongside — including the FINAL (best) val accuracy
hyperparams = {
    "learning_rate":  lr,
    "batch_size":     batch_size,
    "epochs":         epochs,
    "optimizer":      "Adam",
    "scheduler":      "ReduceLROnPlateau(factor=0.5, patience=2)",
    "dropout":        dropout_rate,
    "best_val_acc":   float(best_val_acc),
    "architecture":   "4 conv blocks (32->64->128->256) + GAP + FC(256->128->2)",
}
with open(os.path.join(save_path, 'config_v2.json'), 'w') as f:
    json.dump(hyperparams, f, indent=4)
print("Saved config.")